In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import plotly
import plotly.express as px
import plotly.graph_objects as go

# from google.colab import drive
# drive.mount('/content/drive')

# import os
# os.chdir('/content/drive/MyDrive/DSC106-P2')

economy_df = pd.read_csv("economy-and-growth.csv")
economy_df.head()

environment_df = pd.read_csv("environment.csv")
environment_df.head()

energy_and_mining_df = pd.read_csv('energy-and-mining.csv')
energy_and_mining_df.head()

meta_df = pd.read_csv('Metadata_Country_API_5_DS2_en_csv_v2_3060772.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
economy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15585 entries, 0 to 15584
Data columns (total 53 columns):
 #   Column                                                                                  Non-Null Count  Dtype  
---  ------                                                                                  --------------  -----  
 0   Country Name                                                                            15585 non-null  object 
 1   Country Code                                                                            15585 non-null  object 
 2   Year                                                                                    15585 non-null  int64  
 3   average_value_DEC alternative conversion factor (LCU per US$)                           11078 non-null  float64
 4   average_value_Discrepancy in expenditure estimate of GDP (current LCU)                  7855 non-null   float64
 5   average_value_GDP (constant 2010 US$)                              

# useful columns
- `Country Name`
- `Country Code`
- `Year`
- `average_value_GDP` (constant 2010 U$) everything converted to what a 2010 dollar was worth)


In [5]:
environment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16126 entries, 0 to 16125
Data columns (total 53 columns):
 #   Column                                                                                                    Non-Null Count  Dtype  
---  ------                                                                                                    --------------  -----  
 0   Country Name                                                                                              16126 non-null  object 
 1   Country Code                                                                                              16126 non-null  object 
 2   Year                                                                                                      16126 non-null  int64  
 3   average_value_Adjusted net savings, excluding particulate emission damage (% of GNI)                      7004 non-null   float64
 4   average_value_Adjusted net savings, excluding particulate emission damage (c

# useful columns
- `Country Name`
- `Country Code`
- `Year`
- `Adjusted savings: carbon dioxide damage (% of GNI)`
^^
CO2 pollution causes economic damage — crop losses, health costs, infrastructure wear. This measures that damage as a percentage of a country's income. Useful because it connects environmental harm directly to economics, making it easy to compare rich vs. poor countries fairly regardless of size.


In [6]:
energy_and_mining_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15535 entries, 0 to 15534
Data columns (total 53 columns):
 #   Column                                                                                             Non-Null Count  Dtype  
---  ------                                                                                             --------------  -----  
 0   Country Name                                                                                       15535 non-null  object 
 1   Country Code                                                                                       15535 non-null  object 
 2   Year                                                                                               15535 non-null  int64  
 3   average_value_Access to electricity (% of population)                                              6755 non-null   float64
 4   average_value_Access to electricity, rural (% of rural population)                                 6417 non-null   flo

# Useful Columns
- `Fossil fuel energy consumption (% of total)`
- `Electricity production from renewable sources, excluding hydroelectric (% of total)`
- `Access to electricity (% of population)`
- `average_value_Renewable energy consumption (% of total final energy consumption)`


## Visual 1: Global CO2 Damage Trend
**Proposition:** "Global economic growth has not removed environmental damage over time."

**Summary:** Global CO2 damage as a share of national income stays present across decades.

**Why it matters:** This sets the baseline that environmental harm is persistent and not automatically solved by growth.

### Counter-Argument of Proposition
**Counter-Proposition:** "Economic growth is gradually reducing environmental damage intensity over time."

**Summary:** Even if damage remains visible, cleaner technology and regulation may be lowering damage per unit of output.

**Why it matters:** If true, policy should focus on accelerating efficiency gains rather than framing growth itself as the core problem.

In [49]:
# Line graph: Environmental metric by year (average across all regions)

# Find a suitable environmental metric (e.g., CO2 damage as % of GNI)
metric_col = "average_value_Adjusted savings: carbon dioxide damage (% of GNI)"

# Group by Year only, take mean across all regions
year_metric = environment_df.groupby('Year')[metric_col].mean().reset_index()

# Filter to rows with data
year_metric = year_metric.dropna()

# Line chart: trend over time
fig = px.line(
    year_metric,
    x='Year',
    y=metric_col,
    title='CO2 Damage as % of GNI Over Time',
    labels={metric_col: 'CO2 Damage (% of GNI)'},
    markers=True
)

fig.update_layout(template="plotly_white")
fig.update_yaxes(range=[0.6, 2.5])  # start at ~0.6 not 0
fig.update_layout(title='CO2 Damage Remains Stubbornly High Despite 50 Years of Growth')
fig.show()

print(f"Years covered: {year_metric['Year'].min()} to {year_metric['Year'].max()}")

Years covered: 1970 to 2019


## Visual 2: CO2 Damage by Income Group
**Proposition:** "The environmental cost of growth falls hardest on countries that benefit least."

**Summary:** Lower-income groups face higher CO2-related damage relative to their income.

**Why it matters:** Countries with less wealth have fewer resources to absorb climate and pollution losses.

### Counter-Argument of Proposition
**Counter-Proposition:** "Differences in environmental damage are driven more by domestic policy and energy structure than by income level alone."

**Summary:** Some lower-income countries may perform better than richer peers when policy, urban planning, or energy sources differ.

**Why it matters:** This shifts attention from income category alone to targeted national policy choices.

In [39]:
# Line graph: Environmental metric by year, split by IncomeGroup

# Find a suitable environmental metric (e.g., CO2 damage as % of GNI)
metric_col = "average_value_Adjusted savings: carbon dioxide damage (% of GNI)"

# Merge environment_df with meta_df to get IncomeGroup
env_with_income = environment_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by Year and IncomeGroup, take mean
year_income_metric = env_with_income.groupby(['Year', 'IncomeGroup'])[metric_col].mean().reset_index()

# Filter to rows with data
year_income_metric = year_income_metric.dropna()

# Line chart: one line per income group
fig = px.area(
    year_income_metric,
    x='Year',
    y=metric_col,
    color='IncomeGroup',
    title='CO2 Damage as % of GNI Over Time by Income Group',
    labels={metric_col: 'CO2 Damage (% of GNI)'},
)
fig.update_yaxes(range=[0.5, 5])  # truncated y-axis (deceptive)
fig.update_layout(template='plotly_white')
fig.show()

print(f"Years covered: {year_income_metric['Year'].min()} to {year_income_metric['Year'].max()}")

Years covered: 1970 to 2019


## Visual 3: CO2 Damage by Region
**Proposition:** "Environmental burden is not evenly shared across world regions."

**Summary:** Regional trends show that some parts of the world consistently face higher CO2-related economic damage.

**Why it matters:** Geography and development level shape who bears climate and pollution costs most heavily.

### Counter-Argument of Proposition
**Counter-Proposition:** "Regional differences can overstate inequality because countries within each region are highly diverse."

**Summary:** Regional averages may hide major within-region variation, including both high- and low-damage countries.

**Why it matters:** Strong conclusions should also be validated at country level, not only by regional grouping.

In [40]:
import plotly.express as px

renewable_col = 'average_value_Renewable energy consumption (% of total final energy consumption)'

# Get 2015 data per country
energy_2015 = energy_and_mining_df[energy_and_mining_df['Year'] == 2015][['Country Code', renewable_col]].dropna()

fig = px.choropleth(
    energy_2015,
    locations='Country Code',
    color=renewable_col,
    color_continuous_scale='RdYlGn',  # red=bad, green=good
    title='Renewable Energy Consumption by Country (2015)',
    labels={renewable_col: '% Renewable'},
    range_color=[0, 60]  # cap at 60 so variation at low end looks more extreme (deceptive: hides that many poor countries have high renewables from biomass)
)
fig.update_layout(template='plotly_white')
fig.show()

## Visual 4: Renewable Energy Adoption by Income Group
**Proposition:** "Higher-income countries adopt cleaner energy faster than lower-income countries."

**Summary:** Renewable energy use trends are stronger in richer groups over time.

**Why it matters:** The green transition depends on investment capacity, which poorer countries often lack.

### Counter-Argument of Proposition
**Counter-Proposition:** "Renewable adoption depends on resource endowment and policy design, not only on national income."

**Summary:** Some lower-income countries with strong hydro, solar, or wind potential can scale renewables quickly.

**Why it matters:** International finance and policy support can narrow adoption gaps faster than income growth alone.

In [52]:
# Line graph: Fossil fuel energy consumption by year, split by IncomeGroup

# Metric
metric_col = 'average_value_Fossil fuel energy consumption (% of total)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by Year and IncomeGroup, take mean
year_income_fossil = energy_with_income.groupby(['Year', 'IncomeGroup'])[metric_col].mean().reset_index()

# Filter to rows with data
year_income_fossil = year_income_fossil.dropna()

# Line chart: one line per income group
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'

pivot = year_income_fossil.pivot(index='IncomeGroup', columns='Year', values=fossil_col)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='Reds',
    zmin=30, zmax=90,  # compressed range makes everything look dark red
    colorbar=dict(title='Fossil %'),
))
fig.update_layout(
    title='Fossil Fuel Dependency Over Time by Income Group',
    xaxis_title='Year',
    yaxis_title='Income Group',
    template='plotly_white'
)
fig.show()

print(f"Years covered: {year_income_fossil['Year'].min()} to {year_income_fossil['Year'].max()}")

Years covered: 1960 to 2015


## Visual 5: Fossil Fuel Dependence by Income Group
**Proposition:** "Developing economies remain more tied to fossil fuels during growth."

**Summary:** Fossil fuel shares stay high or rise for many developing income groups.

**Why it matters:** Energy infrastructure paths can lock countries into higher-emission growth models.

### Counter-Argument of Proposition
**Counter-Proposition:** "Fossil fuel dependence in developing economies can be a temporary transition stage rather than a permanent lock-in."

**Summary:** Countries may first expand fossil-based access, then pivot as costs of renewables and storage decline.

**Why it matters:** Transition pathways may be sequential, so short-term fossil use does not automatically imply long-term failure.

In [34]:
# Bar graph: Average Fossil Fuel vs Renewable Energy Consumption by IncomeGroup

# Metrics
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'
renewable_col = 'average_value_Renewable energy consumption (% of total final energy consumption)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by IncomeGroup only, take mean of both metrics
income_energy = energy_with_income.groupby('IncomeGroup').agg({
    fossil_col: 'mean',
    renewable_col: 'mean'
}).reset_index()

# Filter to rows with data
income_energy = income_energy.dropna()

# Reshape data for grouped bar chart
income_energy_melted = income_energy.melt(
    id_vars='IncomeGroup',
    value_vars=[fossil_col, renewable_col],
    var_name='Energy Type',
    value_name='Percentage'
)

# Sort by income level for visual impact
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_energy_melted['IncomeGroup'] = pd.Categorical(
    income_energy_melted['IncomeGroup'], categories=income_order, ordered=True
)
income_energy_melted = income_energy_melted.sort_values('IncomeGroup')

# Rename legend labels to hide that "renewable" for poor countries = biomass
income_energy_melted['Energy Type'] = income_energy_melted['Energy Type'].replace({
    fossil_col: 'Fossil Fuels',
    renewable_col: 'Renewable Energy'  # deceptive: low-income renewable is mostly biomass
})

# Create grouped bar chart
fig = px.bar(
    income_energy_melted,
    x='IncomeGroup',
    y='Percentage',
    color='Energy Type',
    color_discrete_map={fossil_col: '#d62728', renewable_col: '#2ca02c'},
    title='Average Fossil Fuel vs Renewable Energy Consumption by Income Group',
    labels={'Percentage': 'Average Consumption (%)'},
    barmode='stack'
)

fig.update_xaxes(title_text='Income Group')
fig.update_layout(
    template='plotly_white',
    width=1000,
    height=600,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    margin=dict(l=60, r=30, t=110, b=60)
)
fig.show()

print(f"Income Groups: {income_energy['IncomeGroup'].unique()}")

Income Groups: ['High income' 'Low income' 'Lower middle income' 'Upper middle income']


## Visual 6: Energy Mix by Income Group
**Proposition:** "Rich countries use a cleaner energy mix while poorer countries remain more fossil-dependent."

**Summary:** The average energy share shows higher renewable use in richer groups and stronger fossil reliance in poorer groups.

**Why it matters:** Access to cleaner energy technologies is unequal across the global economy.

### Counter-Argument of Proposition
**Counter-Proposition:** "Energy mix differences partly reflect geography, climate, and industrial composition rather than only inequality."

**Summary:** Countries with heavy industry or limited renewable potential may show higher fossil shares regardless of income.

**Why it matters:** Fair comparison should account for structural constraints, not just income grouping.

In [73]:
# Line plot: Fossil and Renewable Energy - Bottom 50% vs Top 50% Income Groups
# Fossil = solid line, Renewable = dashed line

# Metrics
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'
renewable_col = 'average_value_Renewable energy consumption (% of total final energy consumption)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Create income tier grouping: Bottom 50% vs Top 50%
def categorize_income_tier(income_group):
    if income_group in ['Low income', 'Lower middle income']:
        return 'Bottom 50%'
    elif income_group in ['Upper middle income', 'High income']:
        return 'Top 50%'
    else:
        return income_group

energy_with_income['IncomeTier'] = energy_with_income['IncomeGroup'].apply(categorize_income_tier)

# Group by Year and IncomeTier, take mean of both metrics
year_tier_energy = energy_with_income.groupby(['Year', 'IncomeTier']).agg({
    fossil_col: 'mean',
    renewable_col: 'mean'
}).reset_index()

# Filter to rows with data
year_tier_energy = year_tier_energy.dropna()
year_tier_energy = year_tier_energy[year_tier_energy['Year'] >= 1995]

# Create figure
fig = go.Figure()

# Define colors for each tier
colors = {
    'Bottom 50%': '#d62728',
    'Top 50%': '#1f77b4'
}

# Add lines for each tier
for tier in ['Bottom 50%', 'Top 50%']:
    data = year_tier_energy[year_tier_energy['IncomeTier'] == tier]

    # Fossil fuel - solid line
    fig.add_trace(go.Scatter(
        x=data['Year'],
        y=data[fossil_col],
        name=f'{tier} (Fossil)',
        mode='lines',
        line=dict(color=colors.get(tier, '#000000'), width=3, dash='solid'),
        legendgroup=tier,
        hovertemplate='%{fullData.name}<br>Year: %{x}<br>Consumption: %{y:.2f}%<extra></extra>'
    ))

    # Renewable - dashed line
    fig.add_trace(go.Scatter(
        x=data['Year'],
        y=data[renewable_col],
        name=f'{tier} (Renewable)',
        mode='lines',
        line=dict(color=colors.get(tier, '#000000'), width=3, dash='dash'),
        legendgroup=tier,
        hovertemplate='%{fullData.name}<br>Year: %{x}<br>Consumption: %{y:.2f}%<extra></extra>'
    ))


fig.update_layout(
    title='Fossil (solid) vs Renewable (dashed) Energy: Bottom 50% vs Top 50% Income Groups',
    xaxis_title='Year',
    yaxis_title='Energy Consumption (%)',
    template='plotly_white',
    hovermode='x unified',
    height=600
)
fig.add_annotation(x=2010, y=78, text="Top 50%: 78% fossil", showarrow=True, arrowhead=2, font=dict(color='#1f77b4'))
fig.add_annotation(x=2010, y=50, text="Bottom 50%: still rising", showarrow=True, arrowhead=2, font=dict(color='#d62728'))

fig.show()

print(f"Years covered: {year_tier_energy['Year'].min()} to {year_tier_energy['Year'].max()}")

Years covered: 1995 to 2015


## Visual 7: Energy Divide (Bottom 50% vs Top 50%)
**Proposition:** "High-income countries consume more fossil energy and less renewable energy than the Bottom 50%, so the energy transition is uneven"

**Summary:** The Top 50% uses much more fossil energy and much less renewable energy than the Bottom 50%, showing a clear split in energy mix.

**Why it matters:** It shows who is responsible for climate change, who should pay to fix it, and whether the global energy transition is actually happening or just looks that way on paper.

### Counter-Argument of Proposition
**Counter-Proposition:** "The bottom-50 and top-50 grouping may oversimplify energy realities and overstate polarization."

**Summary:** Within each half, countries differ substantially in policy, infrastructure, and transition speed.

**Why it matters:** Broad bins are useful for storytelling but should be complemented by country-level evidence.

In [54]:
# Bar graph: Natural Resource Rents by Income Group

# Metric
metric_col = 'average_value_Total natural resources rents (% of GDP)'

# Merge environment_df with meta_df to get IncomeGroup
env_with_income = environment_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by IncomeGroup, take mean
income_rents = env_with_income.groupby('IncomeGroup')[metric_col].mean().reset_index()

# Filter to rows with data
income_rents = income_rents.dropna()

# Sort by income level (low to high)
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_rents['IncomeGroup'] = pd.Categorical(income_rents['IncomeGroup'], categories=income_order, ordered=True)
income_rents = income_rents.sort_values('IncomeGroup')

# Create bar chart
fig = px.bar(
    income_rents,
    x='IncomeGroup',
    y=metric_col,
    color='IncomeGroup',
    color_discrete_sequence=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
    title='Natural Resource Rents as % of GDP by Income Group',
    labels={metric_col: 'Natural Resource Rents (% of GDP)', 'IncomeGroup': 'Income Group'}
)

fig.update_yaxes(range=[4, 12])
fig.update_layout(title='Poorer Countries Bleed Natural Resources: The Extraction Trap')
fig.show()

print(f"Average Natural Resource Rents by Income Group:")
for _, row in income_rents.iterrows():
    print(f"  {row['IncomeGroup']}: {row[metric_col]:.2f}%")

Average Natural Resource Rents by Income Group:
  Low income: 11.17%
  Lower middle income: 6.98%
  Upper middle income: 6.49%
  High income: 4.55%


## Visual 8: Natural Resource Dependency
**Proposition:** "Low-income countries are more dependent on natural resource rents than rich countries."

**Summary:** Resource rents make up a larger share of GDP in poorer income groups.

**Why it matters:** Heavy resource dependence can increase volatility and slow structural transformation.

### Counter-Argument of Proposition
**Counter-Proposition:** "Resource dependence can be beneficial when revenues are managed well and reinvested productively."

**Summary:** Some countries convert resource rents into infrastructure, education, and stabilization funds.

**Why it matters:** The key issue may be governance quality, not resource dependence itself.

In [66]:
# Line graph: GDP Per Capita Over Time by Income Group (showing widening gap)

# Metric
metric_col = 'average_value_GDP per capita (constant 2010 US$)'

# Merge economy_df with meta_df to get IncomeGroup
economy_with_income = economy_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by Year and IncomeGroup, take mean
year_income_gdp = economy_with_income.groupby(['Year', 'IncomeGroup'])[metric_col].mean().reset_index()

# Filter to rows with data
year_income_gdp = year_income_gdp.dropna()

# Sort by income level for consistent coloring
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
year_income_gdp['IncomeGroup'] = pd.Categorical(year_income_gdp['IncomeGroup'], categories=income_order, ordered=True)

# Create line chart
fig = px.line(
    year_income_gdp,
    x='Year',
    y=metric_col,
    color='IncomeGroup',
    color_discrete_sequence=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
    title='GDP Per Capita Over Time: The Widening Income Gap',
    labels={metric_col: 'GDP Per Capita (constant 2010 US$)'},
    markers=True
)

fig.update_layout(
    title='The Rich Get Richer: 60 Years of Widening Global Inequality',
    meta={'uirevision': True}
)

fig.add_annotation(
    x=2000, y=34000,
    text="High income gained: USD 23,600<br>Low income gained: USD 126",
    showarrow=True, arrowhead=2,
    font=dict(size=12, color='red'),
    bgcolor='white', bordercolor='red',
    align='left',
    borderpad=4,
)
fig.show()

print(f"Years covered: {year_income_gdp['Year'].min()} to {year_income_gdp['Year'].max()}")
print(f"\nGDP Per Capita (constant 2010 US$) by Income Group:")
for income_group in income_order:
    data = year_income_gdp[year_income_gdp['IncomeGroup'] == income_group]
    if len(data) > 0:
        earliest = data[data['Year'] == data['Year'].min()][metric_col].values
        latest = data[data['Year'] == data['Year'].max()][metric_col].values
        if len(earliest) > 0 and len(latest) > 0:
            print(f"  {income_group}:")
            print(f"    {int(data['Year'].min())}: ${earliest[0]:,.0f}")
            print(f"    {int(data['Year'].max())}: ${latest[0]:,.0f}")
            print(f"    Growth: {(latest[0]/earliest[0] - 1)*100:.1f}%")

Years covered: 1960 to 2020

GDP Per Capita (constant 2010 US$) by Income Group:
  Low income:
    1960: $558
    2020: $684
    Growth: 22.5%
  Lower middle income:
    1960: $1,035
    2020: $2,288
    Growth: 121.1%
  Upper middle income:
    1960: $2,432
    2020: $7,038
    Growth: 189.3%
  High income:
    1960: $12,382
    2020: $36,000
    Growth: 190.7%


## Visual 9: The Widening Income Gap
**Proposition:** "High-income countries have grown richer faster, widening the gap between income groups."

**Summary:** GDP per capita trends show faster long-run gains for high-income countries than low-income countries.

**Why it matters:** Persistent divergence weakens the idea that all countries naturally converge with growth.

### Counter-Argument of Proposition
**Counter-Proposition:** "Some lower-income countries are converging, and aggregate group averages can hide their progress."

**Summary:** Fast-growing economies may narrow income gaps even when overall group averages still diverge.

**Why it matters:** Development outcomes are heterogeneous, so policy should identify and replicate successful convergence cases.

In [70]:
# Bar graph: Foreign Income Dependency by Income Group
# Shows remittances and transfers as % of GDP

# Metrics
gdp_col = 'average_value_GDP (current US$)'
foreign_transfers_col = 'average_value_Net secondary income (Net current transfers from abroad) (current US$)'
foreign_income_col = 'average_value_Net primary income (Net income from abroad) (current US$)'

# Merge economy_df with meta_df
economy_with_income = economy_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Use recent year with good data (2018)
recent_year = economy_with_income[economy_with_income['Year'] == 2018].copy()

# Calculate foreign income as % of GDP
recent_year['Foreign_Transfers_pct_GDP'] = (recent_year[foreign_transfers_col] / recent_year[gdp_col]) * 100
recent_year['Foreign_Income_pct_GDP'] = (recent_year[foreign_income_col] / recent_year[gdp_col]) * 100

# Group by IncomeGroup and take mean
income_dependency = recent_year.groupby('IncomeGroup').agg({
    'Foreign_Transfers_pct_GDP': 'mean',
    'Foreign_Income_pct_GDP': 'mean'
}).reset_index()

# Sort by income level
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_dependency['IncomeGroup'] = pd.Categorical(income_dependency['IncomeGroup'], categories=income_order, ordered=True)
income_dependency = income_dependency.sort_values('IncomeGroup')

# Reshape for grouped bar chart
income_dependency_melted = income_dependency.melt(
    id_vars='IncomeGroup',
    value_vars=['Foreign_Transfers_pct_GDP', 'Foreign_Income_pct_GDP'],
    var_name='Foreign Income Type',
    value_name='% of GDP'
)

income_dependency_melted['Foreign Income Type'] = income_dependency_melted['Foreign Income Type'].map({
    'Foreign_Transfers_pct_GDP': 'Remittances & Transfers',
    'Foreign_Income_pct_GDP': 'Primary Income (Investments)'
})

# Create grouped bar chart
fig = px.bar(
    income_dependency_melted,
    x='IncomeGroup',
    y='% of GDP',
    color='Foreign Income Type',
    color_discrete_map={
        'Remittances & Transfers': '#ff7f0e',
        'Primary Income (Investments)': '#1f77b4'
    },
    title='Foreign Income Dependency by Income Group (2018)',
    labels={'IncomeGroup': 'Income Group'},
    barmode='group'
)


fig.update_layout(template='plotly_white')
fig.show()



print(f"Foreign Income Dependency by Income Group (2018):\n")
for income_group in income_order:
    data = income_dependency[income_dependency['IncomeGroup'] == income_group]
    if len(data) > 0:
        transfers = data['Foreign_Transfers_pct_GDP'].values[0]
        primary = data['Foreign_Income_pct_GDP'].values[0]
        total_foreign = transfers + primary
        print(f"  {income_group}:")
        print(f"    Remittances & Transfers: {transfers:.2f}% of GDP")
        print(f"    Primary Income: {primary:.2f}% of GDP")
        print(f"    Total Foreign Income: {total_foreign:.2f}% of GDP")


Foreign Income Dependency by Income Group (2018):

  Low income:
    Remittances & Transfers: 10.66% of GDP
    Primary Income: -1.91% of GDP
    Total Foreign Income: 8.75% of GDP
  Lower middle income:
    Remittances & Transfers: 8.50% of GDP
    Primary Income: 0.93% of GDP
    Total Foreign Income: 9.44% of GDP
  Upper middle income:
    Remittances & Transfers: 5.20% of GDP
    Primary Income: -1.59% of GDP
    Total Foreign Income: 3.61% of GDP
  High income:
    Remittances & Transfers: -0.62% of GDP
    Primary Income: -2.59% of GDP
    Total Foreign Income: -3.20% of GDP


## What This Shows (Visual 9)

*   List item
*   List item



The chart supports the proposition that poorer economies depend more on foreign inflows than richer economies.

**Summary of proposition:** Remittances and external transfers are a much larger share of GDP in low-income groups.

**Why this matters:** When global conditions worsen, countries that depend on outside income face bigger economic risk.

## Visual 10: Access to Electricity Inequality
**Proposition:** "Access to electricity is nearly universal in rich countries but remains a luxury in poor ones."

**Summary:** High-income groups are near universal access, while low-income groups remain far behind.

**Why it matters:** Electricity access is foundational for education, health, jobs, and modern productivity.

### Counter-Argument of Proposition
**Counter-Proposition:** "Electricity access in poorer countries is improving rapidly in many places and may not remain a long-term luxury."

**Summary:** Expansion through grid extension, mini-grids, and off-grid solar can accelerate access faster than historical trends imply.

**Why it matters:** Policy momentum and new technologies can materially shorten the electrification gap.

In [75]:
# Line graph: Access to Electricity Over Time by Income Group

# Metric
metric_col = 'average_value_Access to electricity (% of population)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by Year and IncomeGroup, take mean
year_income_electricity = energy_with_income.groupby(['Year', 'IncomeGroup'])[metric_col].mean().reset_index()

# Filter to rows with data
year_income_electricity = year_income_electricity.dropna()

# Sort by income level for consistent coloring
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
year_income_electricity['IncomeGroup'] = pd.Categorical(year_income_electricity['IncomeGroup'], categories=income_order, ordered=True)

# Create line chart
fig = px.line(
    year_income_electricity,
    x='Year',
    y=metric_col,
    color='IncomeGroup',
    color_discrete_sequence=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
    title='The Electricity Divide: Progress Masked by Averages',
    labels={metric_col: 'Access to Electricity (% of Population)'},
    markers=True
)

fig.update_layout(template="plotly_white", yaxis=dict(range=[0, 105]))
fig.show()

print(f"Years covered: {year_income_electricity['Year'].min()} to {year_income_electricity['Year'].max()}\n")
print(f"Access to Electricity (% of population) by Income Group:\n")
for income_group in income_order:
    data = year_income_electricity[year_income_electricity['IncomeGroup'] == income_group]
    if len(data) > 0:
        earliest = data[data['Year'] == data['Year'].min()][metric_col].values
        latest = data[data['Year'] == data['Year'].max()][metric_col].values
        if len(earliest) > 0 and len(latest) > 0:
            print(f"  {income_group}:")
            print(f"    {int(data['Year'].min())}: {earliest[0]:.1f}%")
            print(f"    {int(data['Year'].max())}: {latest[0]:.1f}%")
            print(f"    Change: +{(latest[0] - earliest[0]):.1f} percentage points\n")


Years covered: 1990 to 2019

Access to Electricity (% of population) by Income Group:

  Low income:
    1990: 32.8%
    2019: 38.0%
    Change: +5.2 percentage points

  Lower middle income:
    1990: 47.1%
    2019: 80.9%
    Change: +33.8 percentage points

  Upper middle income:
    1990: 94.8%
    2019: 96.4%
    Change: +1.6 percentage points

  High income:
    1990: 99.4%
    2019: 100.0%
    Change: +0.6 percentage points



## What This Shows (Visual 10)

The chart confirms a large infrastructure gap between rich and poor countries.

**Summary of proposition:** Wealthier countries already achieved near-100% electricity access, while low-income countries improved slowly and remain far from universal coverage.

**Why this matters:** Limited electricity access blocks long-term progress in human development and economic growth.

## Visual 11: Outsourced Dirty Energy Production
“Higher-income groups still consume the most fossil energy, even as they diversify into renewables.”

**Summary of proposition:** Fossil fuel dependence falls in high-income countries but rises in many middle-income producer countries.

**Why this matters:** Apparent progress in rich countries can overstate global decarbonization if high-emission production is simply relocated.

### Counter-Argument of Proposition
**Counter-Proposition:** "Falling fossil shares in rich countries may reflect genuine decarbonization, not only outsourcing."

**Summary:** Efficiency gains, fuel switching, and domestic policy can reduce fossil dependence even without major production relocation.

**Why it matters:** Correct diagnosis is essential for fair accountability and effective climate policy design.

In [79]:
# Line graph: The Divergence in Fossil Fuel Consumption
# Shows high-income countries DECREASING fossil fuels (outsourcing)
# While middle/low-income countries MAINTAIN or INCREASE fossil fuels (doing the dirty work)

fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by Year and IncomeGroup, take mean
year_income_fossil = energy_with_income.groupby(['Year', 'IncomeGroup'])[fossil_col].mean().reset_index()

# Filter to rows with data
year_income_fossil = year_income_fossil.dropna()

# Sort by income level for consistent coloring
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
year_income_fossil['IncomeGroup'] = pd.Categorical(year_income_fossil['IncomeGroup'], categories=income_order, ordered=True)
year_income_fossil = year_income_fossil[year_income_fossil['Year'] >= 1990]

# Create line chart
fig = px.line(
    year_income_fossil,
    x='Year',
    y=fossil_col,
    color='IncomeGroup',
    color_discrete_sequence=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
    title='Outsourcing Dirty Energy: Rich Countries Decline While Middle-Income Countries Rise',
    labels={fossil_col: 'Fossil Fuel Consumption (% of total energy)'},
    markers=True
)

fig.update_layout(template="plotly_white", height=600)
fig.add_annotation(x=2012, y=57,
    text="↑ Middle-income world absorbs the emissions",
    showarrow=False, font=dict(size=12, color='#2ca02c'),
    bgcolor='white', bordercolor='#2ca02c', borderpad=4, align='left')

fig.add_annotation(x=2012, y=72,
    text="↓ Rich world cleans up its numbers",
    showarrow=False, font=dict(size=12, color='#d62728'),
    bgcolor='white', bordercolor='#d62728', borderpad=4, align='left')
fig.show()

# Calculate trend for each income group
print("Fossil Fuel Consumption Trends by Income Group:\n")
for income_group in income_order:
    data = year_income_fossil[year_income_fossil['IncomeGroup'] == income_group]
    if len(data) > 1:
        start_year = data['Year'].min()
        end_year = data['Year'].max()

        start_value = data[data['Year'] == start_year][fossil_col].values
        end_value = data[data['Year'] == end_year][fossil_col].values

        if len(start_value) > 0 and len(end_value) > 0:
            change = end_value[0] - start_value[0]
            pct_change = (change / start_value[0]) * 100

            trend = "DECREASED ↓" if change < 0 else "INCREASED ↑" if change > 0 else "STABLE →"

            print(f"  {income_group}:")
            print(f"    {int(start_year)}: {start_value[0]:.1f}%")
            print(f"    {int(end_year)}: {end_value[0]:.1f}%")
            print(f"    Change: {change:+.1f} percentage points ({pct_change:+.1f}%) {trend}\n")


Fossil Fuel Consumption Trends by Income Group:

  Low income:
    1990: 34.0%
    2014: 35.3%
    Change: +1.3 percentage points (+3.8%) INCREASED ↑

  Lower middle income:
    1990: 36.0%
    2014: 55.7%
    Change: +19.7 percentage points (+54.9%) INCREASED ↑

  Upper middle income:
    1990: 63.8%
    2015: 88.6%
    Change: +24.8 percentage points (+38.9%) INCREASED ↑

  High income:
    1990: 71.6%
    2015: 68.6%
    Change: -3.0 percentage points (-4.2%) DECREASED ↓

